In [ ]:
# ===============================================
# Depression Classification with Advanced Face Processing
# 주요 기능:
#    - 1 에포크 실행 후 최적의 하이퍼파라미터를 자동으로 조정
#    - 고급 얼굴 크롭 (YUNet + Haar Cascade + 회전 보정)
#    - 검출 실패 샘플 자동 제거 및 캐시 시스템으로 안정성 확보
#    - 명확하게 분리된 3단계 학습 파이프라인 (테스트 -> 조정 -> 본 학습)
# ===============================================

# ===============================================
# STEP 1: 라이브러리 Import
# ===============================================
import os
import random
import json
import hashlib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import resnet50, ResNet50_Weights
import torchvision.transforms as transforms
from PIL import Image, ImageOps, UnidentifiedImageError
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

# OpenCV 안정성 설정
os.environ["OPENCV_OPENCL_RUNTIME"] = "disabled"
cv2.ocl.setUseOpenCL(False)
cv2.setNumThreads(1)
print(f"[OK] OpenCV OpenCL Enabled: {cv2.ocl.useOpenCL()} (False여야 정상)")

In [ ]:
# ===============================================
# STEP 2: 환경 설정 및 하이퍼파라미터
# ===============================================
from pathlib import Path

# 경로 설정 (환경 변수 우선, 없으면 상대 경로 사용)
# 노트북이 hwa_in/ 폴더에 있으므로 현재 디렉토리 기준
_NOTEBOOK_DIR = Path(".").resolve()
BASE_DIR = Path(os.getenv("HWA_IN_DATA_DIR", _NOTEBOOK_DIR / "data"))
MODEL_SAVE_DIR = Path(os.getenv("HWA_IN_MODEL_DIR", _NOTEBOOK_DIR / "model"))
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"[PATH] DATA_DIR: {BASE_DIR}")
print(f"[PATH] MODEL_DIR: {MODEL_SAVE_DIR}")

# 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# 장치 설정
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[DEVICE] Using Device: {DEVICE}")

# 하이퍼파라미터 (1 에포크 후 직접 수정 가능)
HPARAMS = {
    "epochs": 30,
    "batch_size": 64,
    "lr": 1.5e-4,
    "weight_decay": 5e-5,
    "dropout": 0.4,
    "label_smoothing": 0.08,
    "patience": 8,
    "freeze_epochs": 5
}

# 얼굴 검출 파라미터
FACE_CROP_CONFIG = {
    "face_margin": 0.22,
    "min_face_frac": 0.09,
    "tight_mode": True,
    "target_face_fill": 0.85,
    "min_margin_px": 6,
    "use_yunet": True,
    "fail_policy": "skip",
    "validate_on_build": True,
    "cache_dir": str(MODEL_SAVE_DIR / "face_index_cache")
}
Path(FACE_CROP_CONFIG['cache_dir']).mkdir(parents=True, exist_ok=True)
print("[OK] 환경 설정 완료!")

## STEP 3: 데이터셋 클래스 정의

`DepressionDataset` 클래스는 다음 기능을 제공합니다:

1. **얼굴 탐지**: YUNet + Haar Cascade를 사용한 고급 얼굴 검출
2. **캐시 시스템**: 얼굴 검출 결과를 JSON으로 캐시하여 재실행 시 속도 향상
3. **자동 제거**: 얼굴 검출 실패 샘플 자동 제거 (`fail_policy="skip"`)
4. **회전 보정**: 얼굴 랜드마크 기반 기울기 보정

### 주요 메서드
- `_load_samples()`: 이미지 스캔 및 얼굴 탐지
- `_detect_face_bbox()`: YUNet/Haar로 얼굴 영역 검출
- `__getitem__()`: 이미지 로드 및 얼굴 크롭 적용

In [ ]:
# ===============================================
# STEP 3: 데이터셋 클래스 정의
# ===============================================
# 감정 라벨 및 우울증 분류 기준 정의
EMOTIONS = ['anger', 'anxiety', 'hurt', 'joy', 'neutral', 'sadness', 'surprise']
DEP_SET = {'anxiety', 'hurt', 'sadness'}
print(f"[INFO] 우울 감정(1): {DEP_SET}")

# 고급 얼굴 크롭 Dataset (tqdm 진행률 표시줄 추가됨)
class DepressionDataset(Dataset):
    def __init__(self, image_dir, transform=None, **kwargs):
        self.image_dir = image_dir
        self.transform = transform
        
        # kwargs에서 얼굴 검출 설정을 가져옵니다.
        self.face_crop = True
        self.face_margin = float(kwargs.get('face_margin', 0.22))
        self.min_face = tuple(kwargs.get('min_face', (120, 120)))
        self.min_face_frac = float(kwargs.get('min_face_frac', 0.09))
        self.tight_mode = bool(kwargs.get('tight_mode', True))
        self.target_face_fill = float(kwargs.get('target_face_fill', 0.85))
        self.min_margin_px = int(kwargs.get('min_margin_px', 6))
        self.validate_on_build = bool(kwargs.get('validate_on_build', True))
        self.fail_policy = kwargs.get('fail_policy', 'skip')
        self.cache_dir = kwargs.get('cache_dir')
        self.use_yunet = kwargs.get('use_yunet', True) and hasattr(cv2, "FaceDetectorYN")
        self.yunet_path = None # 자동 탐색

        self.samples = []
        self.class_counts = {0: 0, 1: 0}

        # 얼굴 탐지기 관련 변수 초기화
        self._haar_frontal_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        self._haar_profile_path = cv2.data.haarcascades + "haarcascade_profileface.xml"
        self._cascade_frontal = None
        self._cascade_profile = None
        self._yunet = None
        self._yunet_path = self._guess_yunet_path()
        self._face_cache = {}

        # 데이터 로드 메소드 호출
        self._load_samples()

    # --- [수정된 부분] tqdm 진행률 표시줄이 추가된 데이터 로드 메소드 ---
    def _load_samples(self):
        """샘플을 로드하거나 캐시에서 복원합니다."""
        cache_path = self._cache_path()
        if self.validate_on_build and os.path.exists(cache_path):
            with open(cache_path, "r", encoding="utf-8") as file:
                cached_data = json.load(file)
            self.samples = [(path, int(label)) for path, label in cached_data["samples"]]
            self.class_counts = {
                0: int(cached_data["count_non_depressed"]),
                1: int(cached_data["count_depressed"])
            }
            self._face_cache = {
                key: tuple(value) if value is not None else None
                for key, value in cached_data.get("face_cache", {}).items()
            }
            print(f"[CACHE] 캐시에서 빠르게 로드 완료: {os.path.basename(cache_path)} | 샘플={len(self.samples)}")
            return

        print(f"[SCAN] 캐시 파일 없음. 이미지 스캔 및 얼굴 탐지를 시작합니다 (시간이 매우 오래 걸립니다): {self.image_dir}")
        
        all_files_to_scan = []
        for emotion in EMOTIONS:
            emotion_dir = os.path.join(self.image_dir, emotion)
            if not os.path.isdir(emotion_dir):
                continue  # 디렉토리가 없으면 건너뛰기
            
            label = 1 if emotion in DEP_SET else 0
            for filename in os.listdir(emotion_dir):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    all_files_to_scan.append((os.path.join(emotion_dir, filename), label))

        validated_samples = []
        count_non_depressed = 0
        count_depressed = 0
        
        pbar_desc = f"Scanning & Detecting Faces in {os.path.basename(self.image_dir)}"
        for path, label in tqdm(all_files_to_scan, desc=pbar_desc):
            if self.validate_on_build and self.face_crop:
                try:
                    if os.path.getsize(path) < 1024:
                        continue
                    img = Image.open(path).convert("RGB")
                    bbox = self._detect_face_bbox(np.array(self._exif_fix(img)))
                    self._face_cache[path] = bbox
                    
                    if bbox is None and self.fail_policy == "skip":
                        continue
                except Exception:
                    continue

            validated_samples.append((path, label))
            if label == 0:
                count_non_depressed += 1
            else:
                count_depressed += 1
            
        self.samples = validated_samples
        self.class_counts = {0: count_non_depressed, 1: count_depressed}
        print(f"\n[OK] 스캔 완료! 샘플: total={len(self.samples)}, non-dep(0)={count_non_depressed}, dep(1)={count_depressed}")

        if self.validate_on_build:
            cache_data = {
                "samples": self.samples,
                "count_non_depressed": count_non_depressed,
                "count_depressed": count_depressed,
                "face_cache": self._face_cache
            }
            with open(cache_path, "w", encoding="utf-8") as file:
                json.dump(cache_data, file, ensure_ascii=False)
            print(f"[SAVE] 캐시 저장 완료: {os.path.basename(cache_path)}")

    # --- 이하 헬퍼 메소드 및 필수 메소드 (전체 포함) ---
    def _guess_yunet_path(self):
        try:
            cv2_mod_path = cv2.__path__[0]
        except Exception:
            return None
        candidates = [
            os.path.join(cv2_mod_path, 'data', 'face_detection_yunet_2023mar.onnx'),
            os.path.join(cv2_mod_path, 'data', 'face_detection_yunet_2022mar.onnx'),
        ]
        for candidate_path in candidates:
            if os.path.exists(candidate_path):
                return candidate_path
        return None

    def _lazy_load_detectors(self, img_wh=None):
        if self.use_yunet and self._yunet is None and self._yunet_path and os.path.exists(self._yunet_path):
            width, height = img_wh or (320, 320)
            try:
                self._yunet = cv2.FaceDetectorYN.create(
                    self._yunet_path, "", (width, height),
                    score_threshold=0.6, nms_threshold=0.3, top_k=5000
                )
            except Exception:
                self._yunet = None
        if self._cascade_frontal is None:
            self._cascade_frontal = cv2.CascadeClassifier(self._haar_frontal_path)
        if self._cascade_profile is None:
            self._cascade_profile = cv2.CascadeClassifier(self._haar_profile_path)

    @staticmethod
    def _exif_fix(pil_img: Image.Image) -> Image.Image:
        return ImageOps.exif_transpose(pil_img)

    @staticmethod
    def _to_square(x, y, w, h, image_width, image_height, margin=0.3):
        x0 = x - margin * w
        y0 = y - margin * h
        x1 = x + w + margin * w
        y1 = y + h + margin * h
        side = max(x1 - x0, y1 - y0)
        center_x = (x0 + x1) / 2.0
        center_y = (y0 + y1) / 2.0
        x0 = max(0, int(round(center_x - side / 2)))
        y0 = max(0, int(round(center_y - side / 2)))
        x1 = min(image_width, int(round(center_x + side / 2)))
        y1 = min(image_height, int(round(center_y + side / 2)))
        side = min(x1 - x0, y1 - y0)
        return x0, y0, x0 + side, y0 + side

    def _tighten_square(self, x0, y0, x1, y1, face_width, face_height, image_width, image_height):
        if not self.tight_mode:
            return x0, y0, x1, y1
        face_side = max(face_width, face_height)
        fill = np.clip(self.target_face_fill, 0.5, 0.95)
        target_side = max(face_side / fill, face_side + 2 * self.min_margin_px)
        current_side = min(x1 - x0, y1 - y0)
        new_side = min(current_side, target_side)
        center_x = x0 + current_side / 2
        center_y = y0 + current_side / 2
        new_x0 = int(round(center_x - new_side / 2))
        new_y0 = int(round(center_y - new_side / 2))
        new_x1 = new_x0 + int(new_side)
        new_y1 = new_y0 + int(new_side)
        if new_x0 < 0:
            new_x1 -= new_x0
            new_x0 = 0
        if new_y0 < 0:
            new_y1 -= new_y0
            new_y0 = 0
        if new_x1 > image_width:
            new_x0 -= (new_x1 - image_width)
            new_x1 = image_width
        if new_y1 > image_height:
            new_y0 -= (new_y1 - image_height)
            new_y1 = image_height
        side = min(new_x1 - new_x0, new_y1 - new_y0)
        return new_x0, new_y0, new_x0 + side, new_y0 + side

    def _roll_angle_from_landmarks(self, landmarks):
        left_eye_x, left_eye_y = landmarks[0], landmarks[1]
        right_eye_x, right_eye_y = landmarks[2], landmarks[3]
        return np.degrees(np.arctan2(right_eye_y - left_eye_y, right_eye_x - left_eye_x))

    def _detect_face_bbox(self, rgb: np.ndarray):
        height, width = rgb.shape[:2]
        area = width * height
        self._lazy_load_detectors((width, height))
        if self._yunet is not None:
            try:
                self._yunet.setInputSize((width, height))
                ok, faces = self._yunet.detect(rgb)
                if ok and faces is not None and len(faces) > 0:
                    keep = [
                        face for face in faces
                        if face[4] >= 0.6 and face[2] * face[3] >= self.min_face_frac * area
                    ]
                    if keep:
                        best_face = max(keep, key=lambda t: t[2] * t[3])
                        x, y, w, h = map(int, best_face[:4])
                        try:
                            angle = float(self._roll_angle_from_landmarks(best_face[5:9]))
                            return ("lmk", angle, x, y, w, h)
                        except Exception:
                            return (x, y, w, h)
            except Exception:
                pass
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        for detector in (self._cascade_frontal, self._cascade_profile):
            try:
                faces = detector.detectMultiScale(
                    gray, scaleFactor=1.1, minNeighbors=5, minSize=self.min_face
                )
                if len(faces):
                    x, y, w, h = sorted(faces, key=lambda b: b[2] * b[3], reverse=True)[0]
                    if w * h >= self.min_face_frac * area:
                        return (int(x), int(y), int(w), int(h))
            except Exception:
                continue
        return None

    def _cache_key(self):
        payload = {
            "dir": self.image_dir, "face_margin": self.face_margin, 
            "min_face_frac": self.min_face_frac, "tight": self.tight_mode, 
            "fill": self.target_face_fill, "min_margin_px": self.min_margin_px, 
            "use_yunet": self.use_yunet, "validate": self.validate_on_build, 
            "fail_policy": self.fail_policy
        }
        return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()

    def _cache_path(self):
        return os.path.join(
            self.cache_dir,
            f"index_{os.path.basename(self.image_dir)}_{self._cache_key()}.json"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            # 손상된 이미지일 경우, 다음 이미지로 대체
            new_idx = (idx + 1) % len(self.samples)
            path, label = self.samples[new_idx]
            img = Image.open(path).convert("RGB")
        
        img = self._exif_fix(img)

        if self.face_crop:
            bbox = self._face_cache.get(path)
            if bbox is None:
                # 이 경우는 validate_on_build=False 일 때만 발생
                bbox = self._detect_face_bbox(np.array(img))
                self._face_cache[path] = bbox
            
            if bbox is not None:
                img_width, img_height = img.size
                face_width, face_height = bbox[2], bbox[3]  # 원본 bbox 크기 저장
                
                if isinstance(bbox, tuple) and len(bbox) == 6 and bbox[0] == "lmk":
                    _, angle, x, y, w, h = bbox
                    img = self._exif_fix(img.rotate(-angle, expand=True))
                    img_width, img_height = img.size
                else:
                    x, y, w, h = bbox
                    
                x0, y0, x1, y1 = self._to_square(x, y, w, h, img_width, img_height, margin=self.face_margin)
                x0, y0, x1, y1 = self._tighten_square(x0, y0, x1, y1, face_width, face_height, img_width, img_height)
                img = img.crop((x0, y0, x1, y1))
                
            elif self.fail_policy == "center":
                img_width, img_height = img.size
                side = int(min(img_width, img_height) * 0.7)
                center_x, center_y = img_width // 2, img_height // 2
                x0 = max(0, center_x - side // 2)
                y0 = max(0, center_y - side // 2)
                img = img.crop((x0, y0, x0 + side, y0 + side))

        if self.transform:
            img = self.transform(img)
            
        return img, label

    def get_class_weights(self):
        total_samples = sum(self.class_counts.values())
        if total_samples == 0:
            return {0: 1.0, 1: 1.0}
        weight_non_depressed = total_samples / (2.0 * max(1, self.class_counts[0]))
        weight_depressed = total_samples / (2.0 * max(1, self.class_counts[1]))
        return {0: weight_non_depressed, 1: weight_depressed}

## STEP 4: 모델 클래스 정의

`DepressionClassifier`는 **Transfer Learning** 방식으로 우울증을 분류합니다.

### 아키텍처
```
ResNet50 (ImageNet 사전학습)
    └── fc layer 제거 (Identity)
    └── Custom Classifier
        ├── Dropout (과적합 방지)
        └── Linear(2048 → 2)  # 이진 분류
```

### 핵심 메서드
- `freeze_backbone(freeze=True)`: 초기 학습 시 backbone을 고정하여 classifier만 학습
- 이후 `freeze_backbone(freeze=False)`로 전체 fine-tuning

## STEP 5-6: 유틸리티 함수 및 데이터 로드

### 데이터 증강 전략
| Transform | 목적 |
|-----------|------|
| `RandomCrop(224)` | 다양한 위치에서 얼굴 학습 |
| `RandomHorizontalFlip` | 좌우 대칭 증강 |
| `RandomAffine` | 미세 회전/이동으로 로버스트성 향상 |
| `RandomErasing` | Cutout 효과로 과적합 방지 |

### WeightedRandomSampler
클래스 불균형 해소를 위해 **소수 클래스(우울)를 오버샘플링**합니다.

In [ ]:
# ===============================================
# STEP 4: 모델 클래스 정의
# ===============================================
class DepressionClassifier(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity() # 마지막 레이어 제거
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 2)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

    def freeze_backbone(self, freeze=True):
        for name, param in self.backbone.named_parameters():
            if 'fc' not in name: # fc 레이어를 제외한 모든 파라미터
                param.requires_grad = not freeze
print("[OK] 모델 클래스 정의 완료!")

In [ ]:
# ===============================================
# STEP 5: 유틸리티 함수 (데이터 로더, 시각화 등)
# ===============================================

# ImageNet 정규화 상수
# 출처: https://pytorch.org/vision/stable/models.html
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# 이미지 크기 상수
# ResNet/ConvNeXt 등 ImageNet 사전학습 모델의 표준 입력 크기
IMAGE_RESIZE = 256  # Resize 후 크기 (이후 crop)
IMAGE_CROP_SIZE = 224  # 최종 입력 크기 (ImageNet 표준)

# 데이터 변환 정의
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_RESIZE),
    transforms.RandomCrop(IMAGE_CROP_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.2))
])
val_transform = transforms.Compose([
    transforms.Resize(IMAGE_RESIZE),
    transforms.CenterCrop(IMAGE_CROP_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# 데이터 로더 생성 함수
def create_dataloader(dataset, batch_size, is_train=True):
    sampler = None
    if is_train:
        class_weights = dataset.get_class_weights()
        weights = [class_weights[label] for _, label in dataset.samples]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    
    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=(is_train and sampler is None),
        num_workers=4
    )

# 예측 결과 시각화 함수
def visualize_predictions(model, dataloader, device, num_images=8):
    """1 에포크 학습 후 예측 결과를 시각화합니다."""
    model.eval()
    images, labels = next(iter(dataloader))
    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

    images = images.cpu().numpy()
    
    fig = plt.figure(figsize=(20, 5))
    for idx in np.arange(num_images):
        ax = fig.add_subplot(1, num_images, idx + 1, xticks=[], yticks=[])
        # Un-normalize
        img = images[idx]
        img = np.transpose(img, (1, 2, 0))
        img = IMAGENET_STD * img + IMAGENET_MEAN
        img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.set_title(
            f"pred: {preds[idx].item()}\nlabel: {labels[idx].item()}",
            color=("green" if preds[idx] == labels[idx] else "red")
        )
    plt.show()

# =======================================================================
# 얼굴 크롭 결과를 확인하는 함수
# =======================================================================
def show_cropped_faces(dataset, num_samples=5):
    """
    데이터셋의 얼굴 크롭 전처리 결과를 시각적으로 확인하는 함수.
    원본 이미지와 크롭된 이미지를 비교하여 보여줍니다.
    """
    # 시각화를 위해 transform을 일시적으로 비활성화하여 PIL Image를 직접 받습니다.
    original_transform = dataset.transform
    dataset.transform = None

    fig, axes = plt.subplots(2, num_samples, figsize=(18, 7))
    fig.suptitle("얼굴 크롭 전처리 결과 검증", fontsize=16)

    for i in range(num_samples):
        # 데이터셋에서 무작위 샘플 선택
        idx = random.randint(0, len(dataset) - 1)
        path, label = dataset.samples[idx]

        # 1. 원본 이미지 표시
        original_img = Image.open(path).convert("RGB")
        axes[0, i].imshow(original_img)
        axes[0, i].set_title(f"원본 이미지\n(Label: {label})")
        axes[0, i].axis('off')

        # 2. DepressionDataset을 통해 크롭된 이미지 표시
        # __getitem__ 메소드가 얼굴 탐지 및 크롭 로직을 자동으로 수행합니다.
        cropped_img, _ = dataset[idx]  # transform이 None이므로 크롭된 PIL Image 반환
        
        if cropped_img:
            axes[1, i].imshow(cropped_img)
            axes[1, i].set_title("크롭된 얼굴")
        else:  # 얼굴 검출 실패 시 (fail_policy='keep'인 경우 등)
            axes[1, i].imshow(original_img)
            axes[1, i].set_title("얼굴 검출 실패")
        axes[1, i].axis('off')

    # 원래 transform으로 복원
    dataset.transform = original_transform
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

print("[OK] 유틸리티 함수 정의 완료!")

## 학습 파이프라인 (3-Phase Training)

이 노트북은 **3단계 학습 파이프라인**을 사용합니다:

```mermaid
flowchart LR
    P1["PHASE 1<br>1 에포크 테스트"] --> P2["PHASE 2<br>하이퍼파라미터 조정"] --> P3["PHASE 3<br>본 학습"]
```

| Phase | 목적 | Backbone |
|-------|------|----------|
| **PHASE 1** | 데이터/모델 정상 작동 확인 | Frozen |
| **PHASE 2** | 결과 보고 학습률 등 조정 | - |
| **PHASE 3** | Early Stopping 기반 본 학습 | Epoch 5 이후 Unfreeze |

In [ ]:
# ===============================================
# STEP 6: 데이터 로드 및 모델 초기화
# ===============================================
# 데이터셋 인스턴스 생성
train_dataset = DepressionDataset(
    str(BASE_DIR / "train" / "train_image"),
    transform=train_transform,
    **FACE_CROP_CONFIG
)
val_dataset = DepressionDataset(
    str(BASE_DIR / "vali" / "vali_image"),
    transform=val_transform,
    **FACE_CROP_CONFIG
)

# 데이터 로더 생성
train_loader = create_dataloader(train_dataset, HPARAMS["batch_size"], is_train=True)
val_loader = create_dataloader(val_dataset, HPARAMS["batch_size"], is_train=False)

# 모델, 손실함수, 옵티마이저 초기화
model = DepressionClassifier(dropout_rate=HPARAMS["dropout"]).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=HPARAMS["label_smoothing"])
optimizer = optim.AdamW(model.parameters(), lr=HPARAMS["lr"], weight_decay=HPARAMS["weight_decay"])

print("[OK] 데이터 로드 및 모델 초기화 완료!")

In [ ]:
# ===============================================
# STEP 6.5: 전처리 결과 시각적 검증
# ===============================================
print("\n[CHECK] 얼굴 크롭 전처리 결과를 확인합니다...")
print("상단은 원본 이미지, 하단은 DepressionDataset에 의해 자동으로 크롭된 얼굴입니다.")

# 학습 데이터셋의 일부 샘플을 무작위로 뽑아 검증
show_cropped_faces(train_dataset, num_samples=5)

In [ ]:
# ===============================================
# PHASE 1 : 1 에포크 학습 및 시각적 검증
# ===============================================
print("\n PHASE 1: 1 에포크 학습을 시작합니다.")
print("-" * 50)

# Backbone 고정
model.freeze_backbone(freeze=True)
print("[FROZEN] Backbone is frozen.")

# 1 에포크 학습
model.train()
for inputs, labels in tqdm(train_loader, desc="Epoch 1/Training"):
    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
    
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

# 1 에포크 검증
model.eval()
val_loss, val_preds, val_labels = 0, [], []
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Epoch 1/Validation"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        val_preds.extend(preds.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

val_f1 = f1_score(val_labels, val_preds, average='macro')
print(f"\n[Epoch 1 결과] Validation Loss: {val_loss/len(val_loader):.4f}, Validation F1: {val_f1:.4f}")

# 예측 결과 시각화
print("\n[Epoch 1 예측 결과 시각화]")
visualize_predictions(model, val_loader, DEVICE, num_images=8)

In [ ]:
# ===============================================
# 하이퍼파라미터 수동 조정
# ===============================================
print("\n PHASE 2: 하이퍼파라미터 조정을 진행하고 다음 셀을 실행하세요.")
print("현재 학습률:", HPARAMS["lr"])

# --- 예시: 학습률을 변경하고 싶다면 아래 코드의 주석을 해제하고 값을 바꾸세요 ---
# HPARAMS['lr'] = 1e-5
# optimizer = optim.AdamW(model.parameters(), lr=HPARAMS["lr"], weight_decay=HPARAMS["weight_decay"])
# print(f"학습률이 {HPARAMS['lr']}(으)로 변경되었습니다. 옵티마이저가 재생성되었습니다.")

In [ ]:
# ===============================================
# PHASE 3 : 나머지 에포크 학습 진행
# ===============================================
print("\n PHASE 3: 나머지 학습을 시작합니다.")
print("-" * 50)

best_f1 = val_f1
wait_count = 0
best_model_path = None

for epoch in range(2, HPARAMS["epochs"] + 1):
    # Backbone 고정 해제
    if epoch == HPARAMS["freeze_epochs"] + 1:
        model.freeze_backbone(freeze=False)
        print(f"\n[UNFREEZE] Epoch {epoch}: Backbone is unfrozen! Training all layers.")

    # 학습
    model.train()
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/Training"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    # 검증
    model.eval()
    val_loss, val_preds, val_labels = 0, [], []
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch}/Validation"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    current_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f"\n[Epoch {epoch} 결과] Val Loss: {val_loss/len(val_loader):.4f}, Val F1: {current_f1:.4f}")

    # 최고 성능 모델 저장 및 Early Stopping
    if current_f1 > best_f1:
        print(f"[IMPROVED] F1 Score Improved ({best_f1:.4f} -> {current_f1:.4f}). Saving model...")
        best_f1 = current_f1
        wait_count = 0
        best_model_path = MODEL_SAVE_DIR / f"best_model_epoch{epoch}_f1_{best_f1:.4f}.pth"
        torch.save(model.state_dict(), str(best_model_path))
    else:
        wait_count += 1
        if wait_count >= HPARAMS["patience"]:
            print(f"[STOP] Early stopping at epoch {epoch}. Best F1: {best_f1:.4f}")
            break

print("\n[DONE] 모든 학습이 완료되었습니다!")
print(f"[BEST] 최종 최고 F1 Score: {best_f1:.4f}")
print(f"[SAVED] 베스트 모델 저장 경로: {best_model_path}")